<a href="https://colab.research.google.com/github/new2datasci/Deep-learning-architectures/blob/main/Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q torchmetrics==1.4.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.2/869.2 kB 10.6 MB/s eta 0:00:00


In [ ]:
%pip install -q icecream

In [ ]:
import argparse
import os
import pickle
import time

import matplotlib.pyplot as plt
import numpy as np
import PIL
import torch
import torch.backends.cudnn as cudnn
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.parallel
import torch.utils.data
import torchmetrics as tm
import torchvision
import torchvision.datasets as datasets
import torchvision.models as models
import torchvision.transforms as transforms
from icecream import ic
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    classification_report,
)
from sklearn.svm import LinearSVC
from torch.autograd import Variable
from tqdm import tqdm

# Partie 1 : Architecture VGG16

3. Apply the network on several images of your choice

In [ ]:
torchvision.models.VGG16_Weights.IMAGENET1K_V1.transforms()

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

In [ ]:
dog_meme = Image.open("kadaver-vanilla.webp")
bobr = Image.open("bobr.png").convert("RGB")  # 4 channels here
dog = Image.open("dog.jpg")
cat = Image.open("cat.jpg")

vgg16 = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1)
vgg16.eval()
imagenet_classes = pickle.load(
    open("imagenet_classes.pkl", "rb")
)  # chargement du nom des classes


def print_min_max(img):
    img2 = np.array(img)
    print("Min: %.3f, Max: %.3f" % (img2.min(), img2.max()))
    print(img2.shape)
    return img


def manual_norm(img):
    m = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    return (img - m) / std


model_transform_pipeline = torchvision.models.VGG16_Weights.IMAGENET1K_V1.transforms()
to_view_transform_pipeline = transforms.Compose(
    [
        transforms.Lambda(lambda img: img.convert("RGB")),
        transforms.Resize((256, 256), interpolation=Image.BILINEAR),
        transforms.CenterCrop(224),
        transforms.Lambda(lambda img: np.array(img) / 255.0),
    ]
)
# we reproduce the model transformation for plotting

fig = plt.figure(figsize=(16, 9))
# fig.suptitle('')

# create 3x1 subfigs
subfigs = fig.subfigures(nrows=3, ncols=1)
axs = subfigs[0].subplots(nrows=1, ncols=4)
subfigs[0].suptitle("Original images")
for ax, img in zip(axs, [cat, dog, bobr, dog_meme]):
    ax.imshow(img)

axs = subfigs[1].subplots(nrows=1, ncols=4)
subfigs[1].suptitle("Transformed image")
for ax, img in zip(axs, [cat, dog, bobr, dog_meme]):
    x = model_transform_pipeline(img).permute(1, 2, 0)
    ax.imshow(x)

axs = subfigs[2].subplots(nrows=1, ncols=4)
subfigs[2].suptitle("Predictions")
for ax, img in zip(axs, [cat, dog, bobr, dog_meme]):
    x = model_transform_pipeline(img).unsqueeze(0)  # to batch
    y_hat = vgg16(x).squeeze()
    y_hat = torch.nn.functional.softmax(y_hat, dim=0)
    idx = y_hat.argmax()
    txt = (
        imagenet_classes[idx.item()].split(",")[0]
        + f"\n with a probability of \n{y_hat.max().item():.02%}"
    )  # if multiple classes
    ax.tick_params(axis="x", which="both", bottom=False, top=False, labelbottom=False)
    ax.tick_params(axis="y", which="both", right=False, left=False, labelleft=False)
    ax.text(0.5, 0.5, txt, horizontalalignment="center", verticalalignment="center")


FileNotFoundError: [Errno 2] No such file or directory: 'kadaver-vanilla.webp'